# Tutorial 0: Before Starting
关于这个库的相关介绍和使用注意事项请关注README.md。

## 辅助参数生成
目前，我们是用本地文件模拟可信第三方提供的辅助参数，要想生成这些辅助参数，请运行./debug/offline_generators.py，可根据需要修改生成的数量，注意：在运行前不要忘了修改运行路径至计算库根目录。
利用这种方法可以生成如下参数：用于乘法的beaver三元组，三种大小比较方法各自需要的辅助参数：大小比较MSB方法所用的MSB beaver三元组，DICF方法所需的DICF密钥，GROTTO方法所需的前缀DICF密钥。
当然，也可以按照如下代码生成这些密钥。

In [ ]:
from crypto.primitives.beaver.beaver import BeaverOfflineProvider
from crypto.primitives.function_secret_sharing.key_provider import DICFProvider, ParityDICFProvider
from crypto.protocols.most_significant_bit.msb_triple_provider import MSBProvider

num_of_params = 1000000
DICFProvider.generate_keys(num_of_params)
BeaverOfflineProvider().generate_triple_for_parties(num_of_triples=num_of_params, num_of_party=2)
MSBProvider().gen_msb_triples(num_of_params, num_of_party=2)
ParityDICFProvider.generate_keys(num_of_params)

除了这些密钥外，在进行有些运算操作时还需要其他的参数，比如用于矩阵乘法的矩阵beaver三元组，这个三元组和参与运算的矩阵大小有关。这类参数通常和实际运算相关，所以不便提前生成，其生成方案请看后续教程。

## 配置文件
这里解释一下这个库所用的配置文件，为方便起见，我们现阶段将所有的配置都放在了py文件中，配置文件分为基础配置./config/base_configs.py和网络配置./config/network_config.py，在./config包中还有mpc_config.py，这是用于安全多方（三方）计算所使用的配置，目前的应用为聚类操作。
网络配置中主要是关于安全两方计算的两个参与方的地址，端口等信息的配置，可根据实际情况对应修改，也可在创建参与方对象时根据实际情况配置，至于如何配置参与方的套接字相关信息，在后续教程中会有介绍。
现对基础配置做一个介绍，以便后续使用中更换配置以实现不同的操作。

#### base config
主要基础配置如下：
我们的基础运算都是在环上的运算，我们用的环的比特位数用```BIT_LEN```表示，对应的环的大小为```2**BIT_LEN```，可表示的数范围是```[-2 ** (BIT_LEN - 1), 2 ** (BIT_LEN - 1) - 1]```，我们支持的环的```BIT_LEN```为64和32，在./config/base_configs.py中，有几个配置是和```BIT_LEN```有关的，比如环的大小```RING_MAX```, 半环大小```HALF_RING```等，对于这些和```BIT_LEN```有关的配置信息，**请不要改动**，只需要改动```BIT_LEN```

In [ ]:
BIT_LEN = 64

我们库中所有的运算支持CPU和GPU上进行操作，所以可根据自己的电脑配置选择合适的运行“设备”：cpu，cuda(cuda:0, cuda:1)

In [ ]:
DEVICE = 'cuda'

下面是我们用于大小比较的几个配置，我们支持多种方法用于密文大小比较，分别为MSB, FSS, GROTTO，通过修改GE_TYPE可以实现切换，通过理论和实验我们知道MSB方法需要用到多次通信但是计算量比较小，FSS方法中计算次数比较多，但只需要一次通信，GROTTO是优化的FSS方法，因此在网络状况非常好的情况下，选择MSB用于大小比较优于FSS和GROTTO。
而PRG_TYPE是随机数生成器的种类，目前在使用FSS和GROTTO方法时需要使用到随机数。我们支持TMT和AES，TMT是一种指数随机数产生方法，AES方法是基于AES加密算法产生随机数的方法，在数据量较小时，使用AES更加安全高效。

In [ ]:
GE_TYPE = 'MSB'
PRG_TYPE = 'TMT'

当DEBUG为真时，表示调试模式，在调试模式中，我们所使用的辅助参数不具备随机性，也意味着安全性会有所不足，但在这种模式中，可以更便捷且高效地检查运算过程中是否会出现问题，同时运算需要的辅助参数占用存储空间也会比较小
在实际应用中，DEBUG应为False

In [ ]:
DEBUG = True

这部分是参与计算的数的相关配置信息，我们将数分为整型('int')和浮点型('float')，这根据实际运算情况修改DTYPE。
SCALE表示小数部分的规模，整型运算时没有小数部分，所以```int_scale```始终为1，对于64位的浮点数，我们建议使用**65536**（2的16次方）作为浮点数位数，对于32位的浮点数，建议```float_scale```不超过**256**。
对于SCALE可不用修改，如需修改，只可改动```65536```和```127```对应的部分。
注意：当```float_scale```超过建议的数值时，在进行大部分运算时会有很大的概率出错。

In [ ]:
DTYPE = 'float'
float_scale = 65536 if BIT_LEN == 64 else 127
int_scale = 1
SCALE = float_scale if DTYPE == 'float' else int_scale

最后是关于存储路径相关的配置，建议不要改动，当然如要添加新的文件存储路径，可结合实际情况添加

In [ ]:
base_path = './data/{}'.format(BIT_LEN)
model_file_path = base_path + '/NN/'
triple_path = base_path + '/triples_data/'
dpf_key_path = base_path + '/fss_keys/dpf_keys/'
dcf_key_path = base_path + '/fss_keys/dcf_keys/'
dicf_key_path = base_path + '/fss_keys/dicf_keys/'
parity_dicf_key_path = base_path + '/fss_keys/parity_dicf_keys/'
msb_data_path = base_path + '/MSB/triples_data/'